In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


BRONZE_SCHEMA = "football_data_platform.bronze"
SILVER_SCHEMA = "football_data_platform.silver"
SILVER_TABLE = f"{SILVER_SCHEMA}.team_goalkeeping_stats"

LEAGUES = ["bl", "ll", "lo", "pl", "sa"]


def clean_strings(df: DataFrame) -> DataFrame:
    """Trim string columns and convert empty strings to NULL."""

    for column, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column,
                F.when(F.trim(F.col(column)) == "", None)
                .otherwise(F.trim(F.col(column)))
            )

    return df


# ---------------------------------------------------------
# 1. Read Bronze tables
# ---------------------------------------------------------

dataframes = []

for league in LEAGUES:
    table_name = f"{BRONZE_SCHEMA}.fbref_goalkeeping_stats_{league}"

    if not spark.catalog.tableExists(table_name):
        print(f"WARNING: Table does not exist: {table_name}")
        continue

    df = spark.table(table_name)
    df = clean_strings(df)

    dataframes.append(df)


if not dataframes:
    raise ValueError("No Bronze tables were found.")


# ---------------------------------------------------------
# 2. Combine Bronze data
# ---------------------------------------------------------

df_combined = dataframes[0]

for df in dataframes[1:]:
    df_combined = df_combined.unionByName(
        df,
        allowMissingColumns=True
    )


# ---------------------------------------------------------
# 3. Select and standardize columns
# ---------------------------------------------------------

df_silver = df_combined.select(
    F.col("Squad").cast("string").alias("team_name"),
    F.col("#_Pl").cast("int").alias("goalkeepers_used_count"),
    F.col("Performance_GA").cast("int").alias("goals_against"),
    F.col("Performance_GA90").cast("double").alias("goals_against_per_90"),
    F.col("Performance_SoTA").cast("int").alias("shots_on_target"),
    F.col("Performance_Saves").cast("int").alias("saves"),
    F.col("Performance_Save%").cast("double").alias("save_percentage"),
    F.col("Performance_W").cast("int").alias("wins"),
    F.col("Performance_D").cast("int").alias("draws"),
    F.col("Performance_L").cast("int").alias("losses"),
    F.col("Performance_CS").cast("int").alias("clean_sheets"),
    F.col("Performance_CS%").cast("double").alias("clean_sheet_percentage"),
    F.col("Penalty_Kicks_PKatt").cast("int").alias("penalty_kicks_attempted"),
    F.col("Penalty_Kicks_PKA").cast("int").alias("penalty_kicks_attempted_against"),
    F.col("Penalty_Kicks_PKsv").cast("int").alias("penalty_kicks_saved"),
    F.col("Penalty_Kicks_PKm").cast("int").alias("penalty_kicks_missed"),
    F.col("Penalty_Kicks_Save%")
        .cast("double")
        .alias("penalty_kicks_save_percentage"),

    # Business/context columns
    F.col("_season").cast("string").alias("_season"),
    F.col("_league").cast("string").alias("_league"),
    F.col("_stat").cast("string").alias("_stat"),

    # Bronze metadata
    F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),
    F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),
    F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
)


# ---------------------------------------------------------
# 4. Remove duplicates
# ---------------------------------------------------------

df_silver = df_silver.dropDuplicates(
    ["team_name", "_season", "_league"]
)


# ---------------------------------------------------------
# 5. Add Silver metadata
# ---------------------------------------------------------

df_silver = df_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)


# ---------------------------------------------------------
# 6. Basic validation
# ---------------------------------------------------------

null_team_names = df_silver.filter(
    F.col("team_name").isNull()
).count()

if null_team_names > 0:
    print(f"WARNING: NULL team_name: {null_team_names}")

print(f"Silver rows: {df_silver.count()}")


# ---------------------------------------------------------
# 7. Write Silver
# ---------------------------------------------------------

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Created {SILVER_TABLE}")

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def clean_strings(df: DataFrame) -> DataFrame:
    """Trim strings and convert empty strings to NULL."""

    for column, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column,
                F.when(
                    F.trim(F.col(column)) == "",
                    None
                ).otherwise(F.trim(F.col(column)))
            )

    return df

In [0]:
# ---------------------------------------------------------
# 1. Read Bronze, Clean and Combine
# ---------------------------------------------------------

leagues = [
    "bl",
    "ll",
    "lo",
    "pl",
    "sa"
]

dfs = []

for league in leagues:

    table_name = (
        f"football_data_platform.bronze."
        f"fbref_goalkeeping_stats_{league}"
    )

    if not spark.catalog.tableExists(table_name):
        print(f"WARNING: Table does not exist: {table_name}")
        continue

    df = spark.table(table_name)

    df = clean_strings(df)

    dfs.append(df)


# ---------------------------------------------------------
# 2. Combine
# ---------------------------------------------------------

if not dfs:
    raise ValueError("No Bronze tables were found.")

df_combined = dfs[0]

for df in dfs[1:]:
    df_combined = df_combined.unionByName(
        df,
        allowMissingColumns=True
    )

print(f"Combined rows: {df_combined.count()}")


In [0]:
df_combined.display()

In [0]:

# ---------------------------------------------------------
# 4. Standardize data types
# ---------------------------------------------------------

df_combined = df_combined.select(
    F.col("Squad").cast("string").alias("team_name"),
    F.col("#_Pl").cast("int").alias("goalkeepers_used_count"),
    F.col("Performance_GA").cast("int").alias("goals_against"),
    F.col("Performance_GA90").cast("double").alias("goals_against_per_game_90"),
    F.col("Performance_SoTA").cast("int").alias("shots_on_target"),
    F.col("Performance_Saves").cast("int").alias("saves"),
    F.col("Performance_Save%").cast("double").alias("save_percentage"),
    F.col("Performance_W").cast("int").alias("wins"),
    F.col("Performance_D").cast("int").alias("draws"),
    F.col("Performance_L").cast("int").alias("losses"),
    F.col("Performance_CS").cast("int").alias("clean_sheet"),
    F.col("Performance_CS%").cast("double").alias("clean_sheet_percentage"),
    F.col("Penalty_Kicks_PKatt").cast("int").alias("penalty_kicks_attempted"),
    F.col("Penalty_Kicks_PKA").cast("int").alias("penalty_kicks_attempted_against"),
    F.col("Penalty_Kicks_PKsv").cast("int").alias("penalty_kicks_saved"),
    F.col("Penalty_Kicks_PKm").cast("int").alias("penalty_kicks_missed"),
    F.col("Penalty_Kicks_Save%").cast("double").alias("penalty_kicks_save_percentage"),
    F.col("_season").cast("string").alias("_season"),
    F.col("_league").cast("string").alias("_league"),
    F.col("_stat").cast("string").alias("_stat"),
    F.col("_ingested_at").cast("timestamp").alias("_bronze_ingested_at"),
    F.col("_source_file").cast("string").alias("_bronze_source_file"),
    F.col("_source_system").cast("string").alias("source_system"),
)





In [0]:
# ---------------------------------------------------------
# 5. Remove duplicates
# ---------------------------------------------------------

df_team = df_team.dropDuplicates(
    ["team_id"]
)


# ---------------------------------------------------------
# 6. Basic validation
# ---------------------------------------------------------

null_team_ids = df_team.filter(
    F.col("team_id").isNull()
).count()

null_team_names = df_team.filter(
    F.col("team_name").isNull()
).count()

print(f"Rows: {df_team.count()}")
print(f"NULL team_id: {null_team_ids}")
print(f"NULL team_name: {null_team_names}")


# ---------------------------------------------------------
# 7. Add metadata
# ---------------------------------------------------------

df_team = (
    df_team
    .withColumn("_silver_processed_at", F.current_timestamp())
)

In [0]:
  # ---------------------------------------------------------
# 8. Write Silver
# ---------------------------------------------------------
  df_combined.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"football_data_platform.silver.team_goalkeeping_stats")

    print(f"Created football_data_platform.silver.team_goalkeeping_stats")